# Notebook 03 — Prerequisite Integration: From Arrays to a Trainable Token Model

    ## Learning objectives

    - Translate NumPy reference mathematics into PyTorch
- Validate gradients, batches, masking, and checkpoints
- Use a disciplined debugging ladder before larger LLM experiments

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 3.1 Trace one contract across libraries

The safest bridge from numerical foundations to language-model code is a shape-first reference. Begin with integer token IDs shaped batch by time, gather embedding rows, project hidden features to vocabulary logits, shift predictions against next-token labels, and reduce only valid token losses. Implement the same small computation in NumPy and PyTorch with copied weights. Matching forward values isolates framework mechanics from model design. Write every axis, dtype, and valid range beside the interface; many apparent optimization failures are actually transposes, off-by-one shifts, or unintended broadcasting.


In [ ]:
import numpy as np, torch
rng=np.random.default_rng(3); ids=np.array([[0,2,1],[3,1,4]]); emb=rng.normal(size=(5,4)); proj=rng.normal(size=(4,5))
np_logits=emb[ids]@proj; pt_logits=torch.tensor(emb)[torch.tensor(ids)]@torch.tensor(proj); print(np.max(np.abs(np_logits-pt_logits.numpy())),np_logits.shape)


## 3.2 Verify derivatives independently

Autograd is reliable when the computation supplied to it is correct. Select a tiny parameter tensor, compute central finite differences with several step sizes, and compare them with reverse-mode gradients in float64. Large disagreement can reveal a detached tensor, an in-place mutation, a nondifferentiable operation, or a faulty reduction. Gradient checks are expensive and belong on tiny deterministic examples. Afterward, inspect per-parameter gradient existence, finiteness, scale, and update magnitude during training rather than relying only on a declining scalar loss.


In [ ]:
w=torch.tensor([.4,-.7],dtype=torch.float64,requires_grad=True); f=lambda z:(z.sin()*z.square()).sum(); f(w).backward(); eps=1e-6
fd=[]
for i in range(w.numel()):
 p=w.detach().clone(); m=w.detach().clone(); p[i]+=eps; m[i]-=eps; fd.append(((f(p)-f(m))/(2*eps)).item())
print(w.grad.tolist(),fd)


## 3.3 Build the smallest honest data path

A language-model batch needs tokens, attention or validity information, and labels whose ignored positions are explicit. The collator owns padding and batch construction; the objective owns causal shifting and token-weighted reduction. Inspect decoded rows after every transformation. Overfit one tiny batch with dropout disabled and confirm that the target-token loss approaches zero. Then evaluate under inference mode without recording gradients. This test jointly exercises data, model, objective, optimizer, and mode switching while remaining small enough to diagnose exactly.


In [ ]:
from torch import nn
tokens=torch.tensor([[1,2,3,4],[2,3,0,0]]); labels=tokens[:,1:].clone(); labels[1,1:]=-100
model=nn.Sequential(nn.Embedding(5,8),nn.Linear(8,5)); logits=model(tokens[:,:-1]); loss=nn.functional.cross_entropy(logits.reshape(-1,5),labels.reshape(-1),ignore_index=-100,reduction="sum")/(labels!=-100).sum(); print(logits.shape,loss.item())


## 3.4 Reproducibility and the debugging ladder

Fix independent random generators, record configuration and package versions, and save model, optimizer, step, and RNG state. Compare an uninterrupted run with a save-and-resume branch on identical batches. Debug in layers: establish shapes and dtypes, compare a hand reference, overfit one batch, test checkpoint equivalence, then introduce realistic data and only afterward mixed precision or acceleration. Each new feature should have a measured reason and a regression test. This ladder is reused throughout pretraining, fine-tuning, retrieval training, evaluation, and production serving.


In [ ]:
torch.manual_seed(7); state={"model":model.state_dict(),"rng":torch.get_rng_state(),"step":0}; clone=nn.Sequential(nn.Embedding(5,8),nn.Linear(8,5)); clone.load_state_dict(state["model"]); print(all(torch.equal(a,b) for a,b in zip(model.state_dict().values(),clone.state_dict().values())))


## Reference workflow and evidence standard

Treat the notebook as an experiment, not a recipe. State the question, freeze inputs and
success criteria, establish the simplest baseline, change one material factor, and retain raw
outputs needed to diagnose failures. Record model, tokenizer, template, data and code revisions;
hardware and dtype; random seeds; generation or optimization configuration; token counts;
latency and memory; and results by meaningful slice. A demonstration that runs is evidence of
plumbing, not evidence of general capability.

Test boundaries as well as the happy path: empty and maximum-length inputs, malformed records,
multilingual or code text, unavailable dependencies, cancellation, and adversarial content.
Keep credentials in environment or Colab Secrets and never serialize them with artifacts. Pin
remote revisions, review licenses and custom code, validate saved artifacts in a fresh process,
and prefer deterministic validators wherever outputs can be checked mechanically.

Before applying the technique, compare it with prompting, retrieval, a smaller model, or no
model. Report quality together with compute, storage, latency, and operational complexity. Use
held-out data and paired comparisons, disclose uncertainty and negative results, and define a
rollback path. These practices connect low-level understanding to reliable application work.

A useful completion checklist asks four separate questions. Is the mathematical contract clear
enough to predict shapes, masks, reductions, and failure cases? Does the implementation reproduce
a tiny hand-worked or deterministic reference? Does the measured result survive a held-out set,
relevant slices, and an ablation against a simpler baseline? Can another person reload the exact
artifacts and reconstruct the claim from the manifest? Passing only the first two establishes a
tutorial demonstration; passing all four supports an engineering decision. When a result fails,
preserve the counterexample and update the test suite before changing the implementation.

Finally, separate correctness, capability, efficiency, and safety conclusions. A correct
implementation may have weak capability; a capable prototype may be too costly or unsafe to
deploy. Name the population to which each conclusion applies and avoid converting a single
metric into a universal ranking. Track assumptions beside results, especially tokenizer and
template compatibility, data rights, access-control boundaries, and hardware-specific behavior.
Leave exercises with an executable acceptance criterion, a baseline result, and a short written
interpretation. That combination turns exploratory code into cumulative course evidence that can
be revisited when libraries, model families, or deployment engines change.


## 3.5 One-batch overfit as an integration test

Forward parity and gradient checks validate pieces; overfitting one batch validates their composition. Use a repeated deterministic token pattern, disable stochastic layers, and run enough optimizer steps to drive next-token loss far below its initial value. Track both the loss and whether the predicted sequence matches the labels. If it fails, inspect the causal shift, ignored positions, parameter registration, gradient finiteness, learning rate, and whether parameters actually change. This test does not demonstrate generalization. It establishes that the data path, objective, model, backward pass, and optimizer can express and learn at least one controlled example.


In [ ]:
torch.manual_seed(4)
train_tokens=torch.tensor([[1,2,3,1,2,3],[2,3,1,2,3,1]])
mini=nn.Sequential(nn.Embedding(4,12),nn.Linear(12,4)); opt=torch.optim.AdamW(mini.parameters(),lr=.08)
losses=[]
for step in range(120):
 opt.zero_grad(set_to_none=True); out=mini(train_tokens[:,:-1]); batch_loss=nn.functional.cross_entropy(out.reshape(-1,4),train_tokens[:,1:].reshape(-1)); batch_loss.backward(); opt.step(); losses.append(batch_loss.item())
print(losses[0],losses[-1],out.argmax(-1)); assert losses[-1] < losses[0]*.1


## 3.6 Batch-partition invariance

Evaluation loss should describe the same tokens regardless of how examples are partitioned into batches. Averaging batch means violates that contract when batches contain different numbers of eligible labels. Accumulate an unreduced or summed numerator and a valid-token denominator, then divide once. The same principle governs gradient accumulation: scale contributions by the total intended denominator rather than blindly dividing every microbatch mean by the number of microbatches. Create deliberately unequal masks because equal batch sizes can conceal this bug. Later training and evaluation notebooks reuse this numerator/denominator discipline.


In [ ]:
parts=[torch.tensor([.2,.4,.8]),torch.tensor([1.6])]
wrong=sum(x.mean() for x in parts)/len(parts)
correct=sum(x.sum() for x in parts)/sum(x.numel() for x in parts)
direct=torch.cat(parts).mean()
print(wrong.item(),correct.item(),direct.item()); torch.testing.assert_close(correct,direct); assert not torch.isclose(wrong,direct)


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [PyTorch numerical accuracy](https://docs.pytorch.org/docs/stable/notes/numerical_accuracy.html)
- [PyTorch reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html)


## Exercises

    1. Make the NumPy and PyTorch masked losses agree.
2. Overfit a four-sequence corpus and explain every tensor shape.
3. Prove a resumed two-step run matches an uninterrupted run.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
